# What We will Do here....
- 1. How to select appropriate optimizer
- 2. How to select number of Nodes in a layer
- 3. How to select number of layers
- 4. All in all one model

- Dataset Link  https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

In [134]:
import pandas as pd
import numpy as numpy

df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [135]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [136]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [137]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [138]:
X.shape

(768, 8)

In [139]:
X

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

In [140]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=1)

# Without experiments taking nodes, layers and optimizers

In [141]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout

In [142]:
model  = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [143]:
model.fit(X_train, y_train, batch_size=32, epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6352 - loss: 0.6553 - val_accuracy: 0.6688 - val_loss: 0.6347
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6954 - loss: 0.6038 - val_accuracy: 0.7273 - val_loss: 0.5968
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7199 - loss: 0.5716 - val_accuracy: 0.7597 - val_loss: 0.5682
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7296 - loss: 0.5467 - val_accuracy: 0.7597 - val_loss: 0.5461
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7443 - loss: 0.5268 - val_accuracy: 0.7597 - val_loss: 0.5275
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7459 - loss: 0.5108 - val_accuracy: 0.7662 - val_loss: 0.5136
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7655 - loss: 0.4981 - val_accuracy: 0.7792 - val_loss: 0.5030
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7606 - loss: 0.4885 - val_accuracy: 0.77

# 1. How to select appropriate optimizer

In [144]:
# pip install -U keras-tuner

In [145]:
import keras_tuner as kt

In [146]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [147]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [148]:
# find the best optimizer for our problem
tuner.search(X_train,y_train, epochs=5, validation_data=(X_test,y_test))

In [149]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [150]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [151]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7296 - loss: 0.5257 - val_accuracy: 0.7792 - val_loss: 0.4959
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7443 - loss: 0.5099 - val_accuracy: 0.7857 - val_loss: 0.4846
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7573 - loss: 0.4986 - val_accuracy: 0.7857 - val_loss: 0.4779
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7541 - loss: 0.4908 - val_accuracy: 0.8117 - val_loss: 0.4731
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7590 - loss: 0.4835 - val_accuracy: 0.7922 - val_loss: 0.4704
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7638 - loss: 0.4784 - val_accuracy: 0.7857 - val_loss: 0.4692
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7704 - loss: 0.4729 - val_accuracy: 0.8052 - val_loss: 0.4669
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7785 - loss: 0.4688 - val_accuracy: 0

# 2. Number of Nodes/Neurons in a particular layer

In [152]:
def build_model(hp):
  model = Sequential()

  units = hp.Int('units', min_value=8, max_value=128, step=8)

  model.add(Dense(units=units,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [153]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='mydir',
                        project_name='moin')

Reloading Tuner from mydir/moin/tuner0.json


In [154]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [155]:
tuner.get_best_hyperparameters()[0].values

{'units': 48}

In [156]:
model = tuner.get_best_models(num_models=1)[0]

In [157]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7638 - loss: 0.5335 - val_accuracy: 0.8052 - val_loss: 0.5200
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.5089 - val_accuracy: 0.8052 - val_loss: 0.5020
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.4927 - val_accuracy: 0.7987 - val_loss: 0.4890
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4818 - val_accuracy: 0.8052 - val_loss: 0.4799
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4729 - val_accuracy: 0.8117 - val_loss: 0.4755
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4660 - val_accuracy: 0.8052 - val_loss: 0.4736
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4619 - val_accuracy: 0.7987 - val_loss: 0.4710
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4587 - val_accuracy: 0.79

# 3. How to Select Number of Layers

In [158]:
def build_model(hp):
  model = Sequential()

  model.add(Dense(72,activation='relu',input_dim=8)) # 1st layer
  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
     model.add(Dense(72,activation='relu')) # other layers
  model.add(Dense(1,activation='sigmoid')) #output layer

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [159]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,directory='mydir',
                        project_name='naseem')

Reloading Tuner from mydir/naseem/tuner0.json


In [160]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [161]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10}

In [162]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 50 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [163]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.7720 - loss: 0.4634 - val_accuracy: 0.8312 - val_loss: 0.4895
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7866 - loss: 0.4356 - val_accuracy: 0.7987 - val_loss: 0.4881
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7980 - loss: 0.4145 - val_accuracy: 0.8117 - val_loss: 0.4948
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7980 - loss: 0.4227 - val_accuracy: 0.7987 - val_loss: 0.5034
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8094 - loss: 0.3916 - val_accuracy: 0.8052 - val_loss: 0.5352
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8176 - loss: 0.3826 - val_accuracy: 0.7857 - val_loss: 0.5271
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8257 - loss: 0.3640 - val_accuracy: 0.7987 - val_loss: 0.5221
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8404 - loss: 0.3477 - val_accuracy: 0.81

# Now We will Apply All of them in Single Code

In [164]:
def build_model(hp):
  model = Sequential()

  counter=0

  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
     if counter==0:
      model.add(
           Dense(
              hp.Int('units'+str(i),min_value=8, max_value=128, step=8),
              activation=hp.Choice('activation'+str(i), values=['relu','tanh','sigmoid']),
              input_dim=8
          ),
      )
      model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

     else:
      model.add(
          Dense(
              hp.Int('units'+str(i),min_value=8, max_value=128, step=8),
              activation=hp.Choice('activation'+str(i), values=['relu','tanh','sigmoid']),
           ),
      )
      model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
      counter+=1

  model.add(Dense(1,activation='sigmoid'))

  optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [165]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,directory='mydir',
                        project_name='nasir')

In [166]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 11s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6818181872367859
Total elapsed time: 00h 00m 20s


In [167]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 128,
 'activation0': 'relu',
 'dropout0': 0.7,
 'optimizer': 'sgd',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1,
 'units2': 8,
 'activation2': 'relu',
 'dropout2': 0.1}

In [168]:
model = tuner.get_best_models(num_models=1)[0]

In [169]:
model.fit(X_train,y_train,epochs=200,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5733 - loss: 0.6975 - val_accuracy: 0.6688 - val_loss: 0.6659
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5749 - loss: 0.6852 - val_accuracy: 0.6623 - val_loss: 0.6603
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6010 - loss: 0.6748 - val_accuracy: 0.6494 - val_loss: 0.6556
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6417 - loss: 0.6573 - val_accuracy: 0.6494 - val_loss: 0.6515
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6205 - loss: 0.6700 - val_accuracy: 0.6494 - val_loss: 0.6485
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6433 - loss: 0.6528 - val_accuracy: 0.6494 - val_loss: 0.6445
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6596 - loss: 0.6526 - val_accuracy: 0.6494 - val_loss: 0.6416
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6401 - loss: 0.6570 - val_accuracy: 0.649